## Decoder-Only Architecture in a Transformer

Notebook implements an elementary-scale **decoder-only** Transformer pipeline that introduces a token embedding layer, stacking blocks, final normalization, LM-head output projection, weight tying, to output final tensor shape:

$[B,T,V]$, where $V$ is the vocabulary size.

Rough conceptual pipeline:
````text
        Token IDs
         [B, T]
            │
            ↓
     Token Embedding
    W_E : [V, d_model]
            │
            ↓
 Embedded Representations
     [B, T, d_model]
            │
            ↓
  ┌───────────────────┐
  │Transformer Block 1│
  └─────────┬─────────┘
            │
            ↓
            .
            . Multiple stacked blocks
            .
            |
            ↓
  ┌───────────────────┐
  │Transformer Block N│
  └─────────┬─────────┘
            │
            ↓
      Final RMSNorm
            │
            ↓
    Final Representations
       [B, T, d_model]
            │
            ↓
   Vocabulary Projection
    W_E^T : [d_model, V]
      (Weight Tied)
            │
            ↓
    Vocabulary Logits
        [B, T, V]
````

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[2] # cwd up two directories
sys.path.append(str(ROOT))

In [ ]:
import torch
import torch.nn as nn

from src.llm.config import ModelConfig
from src.llm.attention import CausalSelfAttention
from src.llm.layers import RMSNorm, SwiGLU, TransformerBlock

For inspectability purposes, low-scale and practical toy dimensions are to be used.

This time around, the `vocab_size` and `num_layers` parameters serve a more explicit purpose.

In [ ]:
config = ModelConfig(
    vocab_size=1000,
    d_model=8,
    num_heads=2,
    d_ff=32,
    num_layers=3
)

Batch dimension $B$ is now set to an integer value $>1$. \
This means that model will process **multiple** independent sequences at, all in parallel with the use of hardware like **GPUs**.

**Batching** plays an important role as multiple independent sequences can be processed in **parallel**, instead of sequentially, or sequence-by-sequence.

One **caveat** is that batching speeds up compute, provided the hardware has enough **compute power** and **memory** to take advantage of that parallelism.

---

An example set of token IDs will be used to simulate the embedding process.

$B$: amount of independent sequences \
$T$: tokens per sequence

In [ ]:
batch_size = 2 # number of sequences
sequence_length = 4 # number of tokens in each sequence

token_ids = torch.randint(
    0,
    config.vocab_size,
    (batch_size, sequence_length) # [B, T]
)

print(token_ids.shape)

The tokens are then turned into numerical representations in the **learned embedding table**, with shape $[V, d_{model}]$.

A good conceptually way to view this **before** embedding: \
$X:[B,T,\text{features don't exist}]$, where $T$ is the number of token IDs in each batch

Then after embedding: \
$X:[B,T,d_{model}]$, where each token in $T$ is mapped to a **learned** $d_{model}$-dimensional feature vector that numerically represents the token.

---

Before embedding, tokens are discrete IDs with no feature representation. The embedding layer maps each ID to a learned $d_{model}$-dimensional feature vector that the Transformer can operate on.

In [ ]:
token_embedding = nn.Embedding(
    config.vocab_size, # number of embeddings
    config.d_model # embedding dimension
)

x = token_embedding(token_ids)

print(x.shape)

After embedding, the token representations are passed **sequentially** through multiple Transformer blocks. \
Because data going out of a block is the input for the next, the procedure is called **stacking blocks**:

$ X_0 \rightarrow \text{Block}_1 \rightarrow \text{Block}_2 \rightarrow \cdots \rightarrow \text{Block}_N \rightarrow X_N $, where $N$ is determined by `config.num_layers`.

---

Each block has the **same** architecture defined in `TransformerBlock`, but have **distinct** learned parameters, allowing each layer to progressively refine the token's representation in **different** ways.

Each block **preserves** input/ouput shape:

$X_0:[B,T,d_{model}] \rightarrow ... \rightarrow X_N:[B,T,d_{model}] \rightarrow Y:[B,T,d_{model}]$

`ModuleList` stores multiple PyTorch modules in a list while registering their parameters as part of the model.
https://docs.pytorch.org/docs/2.14/generated/torch.nn.ModuleList.html

In [ ]:
blocks = nn.ModuleList(
    [
        TransformerBlock(config)
        for _ in range(config.num_layers) # number of blocks
    ]
)

for block in blocks:
    x = block(x)

print(x.shape)

Each transformer block ends with a **residual connection**. A final **RMSNorm** is applied before going into the LM head.

This final normalization serves the same fundamental normalization **operation** and **purpose** as the pre-norms in a transformer block.

---

Pre-norms support stable processing within each block, while the **final normalization** ensures the accumulated **final representation is normalized** before producing logits.

In [ ]:
final_norm = RMSNorm(config.d_model)

x = final_norm(x)

print(x.shape)

In order to output a logit for each possible vocabulary token, a final projection is performed:

$Y:[B,T,d_{model}] \times W_LM:[d_{model}, V] \rightarrow [B,T,V]$

---

While it may seem confusing at first thinking how the representations $[B,T,d_{model}]$ all of a sudden turns into scores $[B,T,V]$, it is thought that $W_LM$ learns how combinations of those **features** should **map to vocabulary logits**.

A learned linear projection from $d_{model}$ features to vocabulary $V$ logits. Through **training** cycles, the Transformer learns useful token features while $W_{LM}$ learns **how those features map to scores for each vocabulary token**.

---

Since we are using weight tying in order to streamline and optimize parameter count, we use the **embedding matrix** from earlier, now **transposed** shape $[d_{model}, V]$ in order to project, then output the final logits tensor $[B,T,V]$.

In [ ]:
logits = x @ token_embedding.weight.T

print(logits.shape)